In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
from yellowbrick.cluster import KElbowVisualizer
import matplotlib.pyplot as plt
import pandas as pd 
import seaborn as sns
from sksurv.base import SurvivalAnalysisMixin as s
from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score
from sksurv.preprocessing import encode_categorical
from sksurv.datasets import load_gbsg2
from sksurv.functions import StepFunction
from sksurv.linear_model import CoxPHSurvivalAnalysis, CoxnetSurvivalAnalysis
from sksurv.ensemble import (ComponentwiseGradientBoostingSurvivalAnalysis, 
                            RandomSurvivalForest, 
                            ExtraSurvivalTrees, 
                            GradientBoostingSurvivalAnalysis, 
                            ExtraSurvivalTrees)
from sksurv.meta import EnsembleSelection, EnsembleSelectionRegressor
from sksurv.metrics import integrated_brier_score
from matplotlib.colors import ListedColormap
from mlxtend.evaluate import paired_ttest_5x2cv
from mlxtend.evaluate import combined_ftest_5x2cv
from lifelines import KaplanMeierFitter
from scipy.cluster import hierarchy
from lifelines.statistics import logrank_test, multivariate_logrank_test, pairwise_logrank_test
from sklearn import preprocessing
from sklearn.model_selection import StratifiedKFold, KFold
from lifelines.plotting import add_at_risk_counts
import scipy.stats
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import optuna
from sklearn.model_selection import cross_val_score
from sksurv.metrics import integrated_brier_score
from lifelines import CoxPHFitter
from lifelines.statistics import proportional_hazard_test
import scipy.stats as stats
from statsmodels.stats.outliers_influence import variance_inflation_factor 
import statsmodels.api as sm
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'
from lifelines import CoxPHFitter
from sklearn.preprocessing import RobustScaler

In [2]:
# OUS: Train data
OUS_D1 = pd.read_csv('OUS_D1.csv')
OUS_D2 = pd.read_csv('OUS_D2.csv')
OUS_D3 = pd.read_csv('OUS_D3.csv')
OUS_DFS_target = pd.read_csv('OUS_DFS_target.csv')
OUS_OS_target = pd.read_csv('OUS_OS_target.csv')
response_OUS = pd.read_csv('response_ous.csv', sep=';')

# MAASTRO: Test data 
MAASTRO_D1 = pd.read_csv('MAASTRO_D1.csv')
MAASTRO_D2 = pd.read_csv('MAASTRO_D2.csv')
MAASTRO_D3 = pd.read_csv('MAASTRO_D3.csv')
MAASTRO_DFS_target = pd.read_csv('MAASTRO_DFS_target.csv')
MAASTRO_OS_target = pd.read_csv('MAASTRO_OS_target.csv')
response_MAASTRO = pd.read_csv('maastro_response_full.csv', sep=',')

In [3]:
response_OUS.isna().sum().sum()

0

In [4]:
# Need to choose patient_id from OUS_D3 in response_OUS
data = list(OUS_D3['patient_id'])
mask = response_OUS['patient_id'].isin(data)
response_OUS = response_OUS[mask] 

# Merge OUS_D3 with response_OUS
clinical_train = pd.merge(OUS_D3, response_OUS, on='patient_id', how='inner')
clinical_train = clinical_train.loc[:, ~clinical_train.columns.isin(['OS', 'event_OS', 'LRC', 'event_LRC'])]

In [5]:
# Drop patient_id column
clinical_train = clinical_train.drop('patient_id', axis=1)

In [6]:
# Check null values in D3
clinical_train.isnull().sum().sum()

0

In [7]:
clinical_train

,age,female,cavum_oris,oropharynx,hypopharynx,larynx,histgrade_high,hpv_related,charlson,pack_years,...,LBP_021_PET,LBP_030_PET,LBP_102_PET,LBP_111_PET,LBP_120_PET,LBP_201_PET,LBP_210_PET,LBP_300_PET,DFS,event_DFS
0,54.238356,1,0,1,0,0,1,0.0,0,0.000000,...,0.029974,0.815962,0.000000,0.002035,0.140311,0.000062,0.009991,0.000370,32.909589,0
1,54.539726,0,0,0,0,1,0,0.0,1,27.404795,...,0.059728,0.707300,0.000000,0.008732,0.191058,0.000349,0.023402,0.000699,27.715068,0
2,59.019178,0,1,0,0,0,1,0.0,1,41.019178,...,0.039463,0.819828,0.000034,0.002518,0.126531,0.000000,0.009452,0.000414,43.101370,1
3,70.726027,0,0,0,0,1,0,0.0,1,37.500000,...,0.061133,0.716212,0.000000,0.003296,0.192388,0.000000,0.018879,0.000899,17.589041,1
4,67.865753,0,0,0,0,1,0,0.0,1,53.000000,...,0.058589,0.696493,0.000199,0.008968,0.202073,0.000399,0.029892,0.001395,47.473973,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
134,60.435616,0,0,1,0,0,1,1.0,0,0.000000,...,0.029444,0.804831,0.000000,0.001442,0.152626,0.000000,0.009734,0.000361,41.490411,0
135,68.794521,0,0,1,0,0,1,1.0,0,0.000000,...,0.046451,0.792151,0.000000,0.004094,0.142778,0.000079,0.011810,0.000630,36.821918,0
136,57.498630,0,0,1,0,0,1,1.0,1,39.498630,...,0.016316,0.832202,0.000000,0.000914,0.140582,0.000000,0.009137,0.000522,40.043836,0
137,65.684932,0,0,1,0,0,1,1.0,1,71.527397,...,0.038862,0.787588,0.000000,0.003144,0.156640,0.000054,0.011978,0.000705,40.964384,0


## Test dataset: MAASTRO 

In [8]:
(MAASTRO_D3['patient_id'] == MAASTRO_OS_target['patient_id']).sum()

99

In [9]:
# Rename the column name of response_MAASTRO 
response_MAASTRO.rename(columns = {'Index' : 'patient_id'}, inplace = True)

In [10]:
rows_with_nan = response_MAASTRO[response_MAASTRO.isna().any(axis=1)]

print("Rows with NaN values:")
print(rows_with_nan)

Rows with NaN values:
    patient_id  OS  OS_event  LRC  LRC_event  DFS  DFS_event
10          11 NaN       NaN  NaN        NaN  NaN        NaN
20          21 NaN       NaN  NaN        NaN  NaN        NaN
31          32 NaN       NaN  NaN        NaN  NaN        NaN
51          52 NaN       NaN  NaN        NaN  NaN        NaN
83          84 NaN       NaN  NaN        NaN  NaN        NaN
86          87 NaN       NaN  NaN        NaN  NaN        NaN


In [11]:
# Assuming your data is stored in a list or a pandas DataFrame/Series
# Convert data to a set for faster membership checking
data_set = set(list(response_MAASTRO['patient_id']))

# Find numbers from 1 to 114 not present in data
missing_numbers = set(range(1, 115)) - data_set

# Convert missing_numbers back to a sorted list
missing_numbers_list = sorted(missing_numbers)

In [12]:
MAASTRO_D3

,patient_id,age,female,cavum_oris,oropharynx,hypopharynx,larynx,histgrade_high,hpv_related,charlson,...,LBP_003_PET,LBP_012_PET,LBP_021_PET,LBP_030_PET,LBP_102_PET,LBP_111_PET,LBP_120_PET,LBP_201_PET,LBP_210_PET,LBP_300_PET
0,1,55,0,0,1,0,0,1,1,1,...,0.000026,0.001181,0.028808,0.837387,0.000026,0.001705,0.122209,0.000026,0.008291,0.000341
1,2,55,0,0,1,0,0,0,0,0,...,0.000056,0.002735,0.049615,0.806842,0.000167,0.002958,0.128976,0.000167,0.008148,0.000335
2,3,55,0,0,1,0,0,0,0,1,...,0.000286,0.001888,0.019514,0.830272,0.000057,0.001831,0.137282,0.000000,0.008641,0.000229
3,4,61,1,0,0,0,1,1,0,1,...,0.000160,0.003037,0.047155,0.760949,0.000000,0.003597,0.171595,0.000080,0.013187,0.000240
4,6,70,0,0,1,0,0,1,1,1,...,0.000071,0.000881,0.033990,0.824865,0.000000,0.002116,0.128134,0.000035,0.009379,0.000529
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94,110,66,1,0,0,0,1,0,0,0,...,0.000000,0.000544,0.017489,0.834590,0.000000,0.001321,0.137194,0.000078,0.008706,0.000078
95,111,63,0,0,0,0,1,0,0,1,...,0.000109,0.000869,0.029979,0.821431,0.000000,0.000543,0.137620,0.000054,0.008907,0.000489
96,112,63,0,0,1,0,0,1,1,1,...,0.000000,0.000734,0.017354,0.859957,0.000000,0.000988,0.115099,0.000028,0.005700,0.000141
97,113,54,0,0,1,0,0,1,1,0,...,0.000114,0.001640,0.025399,0.847908,0.000000,0.001487,0.117654,0.000000,0.005759,0.000038


In [13]:
# Assuming your data is stored in a list or a pandas DataFrame/Series
# Convert data to a set for faster membership checking
data_set = set(list(MAASTRO_D3['patient_id']))

# Find numbers from 1 to 114 not present in data
missing_numbers = set(range(1, 115)) - data_set

# Convert missing_numbers back to a sorted list
missing_numbers_list = sorted(missing_numbers)

print("Numbers from 1 to 114 not present in data:", missing_numbers_list)


Numbers from 1 to 114 not present in data: [5, 9, 11, 21, 32, 36, 46, 52, 65, 76, 78, 81, 84, 87, 92]


In [14]:
# Assuming your data is stored in a list or a pandas DataFrame/Series
# Convert data to a set for faster membership checking
data_set = set(list(response_MAASTRO['patient_id']))

# Find numbers from 1 to 114 not present in data
missing_numbers = set(range(1, 115)) - data_set

# Convert missing_numbers back to a sorted list
missing_numbers_list = sorted(missing_numbers)

print("Numbers from 1 to 114 not present in data:", missing_numbers_list)


Numbers from 1 to 114 not present in data: []


In [15]:
# need to choose patient_id from MAASTRO_D3 in response_MAASTRO
data = list(MAASTRO_D3['patient_id'])
mask = response_MAASTRO['patient_id'].isin(data)
response_MAASTRO = response_MAASTRO[mask] 

In [16]:
# Merge MAASTRO_D3 with response_MAASTRO
clinical_test = pd.merge(MAASTRO_D3, response_MAASTRO, on='patient_id', how='inner')
clinical_test = clinical_test.loc[:, ~clinical_test.columns.isin(['OS', 'OS_event', 'LRC', 'LRC_event'])]

In [17]:
# Drop patient_id column
clinical_test = clinical_test.drop('patient_id', axis=1)

In [18]:
# Check if some rows have null values in OS, OS_event -> Remove those rows
clinical_test[clinical_test.isnull().any(axis=1)]
clinical_test = clinical_test.dropna(how='any',axis=0) 

,age,female,cavum_oris,oropharynx,hypopharynx,larynx,histgrade_high,hpv_related,charlson,pack_years,...,LBP_021_PET,LBP_030_PET,LBP_102_PET,LBP_111_PET,LBP_120_PET,LBP_201_PET,LBP_210_PET,LBP_300_PET,DFS,DFS_event


In [19]:
# X
X = clinical_train.loc[:, ~clinical_train.columns.isin(['DFS', 'event_DFS'])]

# y 
y = clinical_train.loc[:, ['DFS', 'event_DFS']]

In [20]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

# Shape
print('X_train: ', X.shape)
print('y_train: ', y.shape)

clinical_test.rename(columns = {'DFS_event' : 'event_DFS'}, inplace = True)

# X
X_MAASTRO = clinical_test.loc[:, ~clinical_test.columns.isin(['DFS', 'event_DFS'])]

# y y_MAASTRO
y_MAASTRO = clinical_test.loc[:, ['DFS', 'event_DFS']]
lower, upper = np.percentile(y_MAASTRO['DFS'], [10, 90])
times = np.arange(lower, upper)

# y into array 
lists = [] 
for i, j in zip(y_MAASTRO['event_DFS'], y_MAASTRO['DFS']): 
    lists.append((i, j))

y_MAASTRO = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

clinical_test.shape

X_train:  (139, 388)
y_train:  (139,)


(99, 390)

# Feature Selection: RENT

In [21]:
selected_features = ["shape_Sphericity",
"hpv_related",
"glszm_SmallAreaLowGrayLevelEmphasis_CT_c16",
"LBP_102_PET",
"shape_Flatness",
"shape_Elongation",
"uicc8_III-IV",
"shape_MajorAxisLength",
"glrlm_HighGrayLevelRunEmphasis_PET_c04"]


# Selecting features in the DataFrame
X_rent = X[selected_features]
X_new = X_rent.copy()

X_MAASTRO_rent = X_MAASTRO[selected_features]
MAASTRO_new = X_MAASTRO_rent.copy()

# Standardization

In [22]:
original_X = X.copy()

In [23]:
# Standardize X_new, the new data with the selected features only 
categorical_columns = ['female', 
                        'cavum_oris',
                        'oropharynx',
                        'hypopharynx',
                        'larynx',
                        'histgrade_high',
                        'hpv_related',
                        'charlson',
                        'uicc8_III-IV']

# Set the columns_to_drop which are categorical 
columns_to_drop = [col for col in X_new.columns if col in categorical_columns]

# Drop the columns if they exist in the DataFrame and save the numeric part in X_new_numeric
X_new_categoric = X_new[columns_to_drop]
X_new_numeric = X_new.drop(columns=columns_to_drop, inplace=False)

# Do the standardization for the numeric part 
scaler = RobustScaler() 
X_new_numeric_columns = X_new_numeric.columns
X_new_numeric_index = X_new_numeric.index 
X_new_numeric_std = scaler.fit_transform(X_new_numeric)
X_new_numeric_std = pd.DataFrame(X_new_numeric_std,
                                 columns=X_new_numeric_columns, 
                                 index=X_new_numeric_index)
X_new_std = pd.concat([X_new_numeric_std, X_new_categoric], axis=1)

# Change the order of the X_new_std 
X_new_std = X_new_std[X_new.columns]

In [24]:
# Standardize X_MAASTRO 
# Set the columns_to_drop which are categorical 
columns_to_drop = [col for col in MAASTRO_new.columns if col in categorical_columns]

# Drop the columns if they exist in the DataFrame and save the numeric part in X_new_numeric
MAASTRO_new_categoric = MAASTRO_new[columns_to_drop]
MAASTRO_new_numeric = MAASTRO_new.drop(columns=columns_to_drop, inplace=False)

# Do the standardization for the numeric part 
MAASTRO_new_numeric_columns = MAASTRO_new_numeric.columns
MAASTRO_new_numeric_columns = MAASTRO_new_numeric.columns
MAASTRO_new_numeric_index = MAASTRO_new_numeric.index 
MAASTRO_new_numeric_std = scaler.transform(MAASTRO_new_numeric)
MAASTRO_new_numeric_std = pd.DataFrame(MAASTRO_new_numeric_std,
                                 columns=MAASTRO_new_numeric_columns, 
                                 index=MAASTRO_new_numeric_index)
MAASTRO_new_std = pd.concat([MAASTRO_new_numeric_std, MAASTRO_new_categoric], axis=1)

# Change the order of the X_new_std 
MAASTRO_new_std = MAASTRO_new_std[MAASTRO_new.columns]

In [25]:
X_new

,shape_Sphericity,hpv_related,glszm_SmallAreaLowGrayLevelEmphasis_CT_c16,LBP_102_PET,shape_Flatness,shape_Elongation,uicc8_III-IV,shape_MajorAxisLength,glrlm_HighGrayLevelRunEmphasis_PET_c04
0,0.761164,0.0,0.029425,0.000000,0.535140,0.600926,0.0,42.073251,16.969770
1,0.697049,0.0,0.037915,0.000000,0.367109,0.841579,0.0,24.613845,15.598394
2,0.565792,0.0,0.008009,0.000034,0.597785,0.772821,1.0,48.030294,17.334294
3,0.684364,0.0,0.018398,0.000000,0.405730,0.847727,0.0,25.589900,14.009277
4,0.503142,0.0,0.013051,0.000199,0.442406,0.831483,0.0,34.684750,21.202180
...,...,...,...,...,...,...,...,...,...
134,0.742102,1.0,0.014938,0.000000,0.523608,0.680294,0.0,33.069705,16.110383
135,0.722918,1.0,0.011441,0.000000,0.735524,0.758193,1.0,41.043692,21.249575
136,0.652963,1.0,0.020431,0.000000,0.648063,0.770113,0.0,36.618802,14.873884
137,0.724255,1.0,0.019663,0.000000,0.492193,0.628897,1.0,45.870392,21.648860


In [26]:
X_new_std

,shape_Sphericity,hpv_related,glszm_SmallAreaLowGrayLevelEmphasis_CT_c16,LBP_102_PET,shape_Flatness,shape_Elongation,uicc8_III-IV,shape_MajorAxisLength,glrlm_HighGrayLevelRunEmphasis_PET_c04
0,0.704475,0.0,1.172200,0.000000,0.064971,-0.485459,0.0,0.059912,0.085656
1,0.101791,0.0,1.947696,0.000000,-0.952446,0.666232,0.0,-0.755402,-0.291967
2,-1.132016,0.0,-0.783916,0.524263,0.444285,0.337179,1.0,0.338093,0.186031
3,-0.017443,0.0,0.165035,0.000000,-0.718597,0.695653,0.0,-0.709822,-0.729547
4,-1.720923,0.0,-0.323382,3.028668,-0.496529,0.617917,0.0,-0.285114,1.251094
...,...,...,...,...,...,...,...,...,...
134,0.525285,1.0,-0.151010,0.000000,-0.004852,-0.105628,0.0,-0.360532,-0.150985
135,0.344965,1.0,-0.470412,0.000000,1.278289,0.267171,1.0,0.011834,1.264145
136,-0.312609,1.0,0.350720,0.000000,0.748713,0.324219,0.0,-0.194798,-0.491468
137,0.357529,1.0,0.280586,0.000000,-0.195069,-0.351598,1.0,0.237230,1.374092


In [27]:
MAASTRO_new

,shape_Sphericity,hpv_related,glszm_SmallAreaLowGrayLevelEmphasis_CT_c16,LBP_102_PET,shape_Flatness,shape_Elongation,uicc8_III-IV,shape_MajorAxisLength,glrlm_HighGrayLevelRunEmphasis_PET_c04
0,0.668072,1,0.010495,0.000026,0.610062,0.765178,0,50.002093,19.034156
1,0.669961,0,0.035018,0.000167,0.504616,0.776540,1,41.753334,11.392444
2,0.624081,0,0.012819,0.000057,0.478604,0.697164,1,44.375483,14.567421
3,0.577624,0,0.029974,0.000000,0.446059,0.574636,1,46.115989,13.477331
4,0.630933,1,0.013458,0.000000,0.480378,0.633419,0,54.394967,16.365554
...,...,...,...,...,...,...,...,...,...
94,0.671754,0,0.039092,0.000000,0.577884,0.882411,1,34.218615,17.492295
95,0.632189,0,0.015831,0.000000,0.455642,0.535802,1,51.046869,14.267900
96,0.645548,1,0.011031,0.000000,0.631485,0.716610,1,50.417953,12.143752
97,0.727488,1,0.019801,0.000000,0.628338,0.665145,0,44.901412,15.300229


In [28]:
MAASTRO_new_std

,shape_Sphericity,hpv_related,glszm_SmallAreaLowGrayLevelEmphasis_CT_c16,LBP_102_PET,shape_Flatness,shape_Elongation,uicc8_III-IV,shape_MajorAxisLength,glrlm_HighGrayLevelRunEmphasis_PET_c04
0,-0.170593,1,-0.556880,0.398737,0.518623,0.300600,0,0.430171,0.654106
1,-0.152832,0,1.683040,2.544568,-0.119851,0.354974,1,0.044973,-1.450119
2,-0.584101,0,-0.344618,0.869691,-0.277351,-0.024893,1,0.167421,-0.575856
3,-1.020797,0,1.222314,0.000000,-0.474407,-0.611274,1,0.248699,-0.876024
4,-0.519690,1,-0.286171,0.000000,-0.266607,-0.329957,0,0.635308,-0.080721
...,...,...,...,...,...,...,...,...,...
94,-0.135979,0,2.055216,0.000000,0.323784,0.861639,1,-0.306881,0.229539
95,-0.507883,0,-0.069508,0.000000,-0.416383,-0.797117,1,0.478960,-0.658333
96,-0.382318,1,-0.507885,0.000000,0.648334,0.068172,1,0.449591,-1.243239
97,0.387916,1,0.293176,0.000000,0.629283,-0.178126,0,0.191981,-0.374070


# Modelling 

### 1. CoxPHSurvivalAnalysis

#### Train

In [29]:
# Setting the y format for skf below  
y = clinical_train[['DFS', 'event_DFS']]

# Running to optuna for hyperparameter tuning 
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class()
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Robust Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = RobustScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxPHSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxPHSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-19 14:31:26,313] A new study created in memory with name: no-name-e1d9ef13-4d17-4956-8d33-b511ece8908b


  0%|          | 0/1 [00:00<?, ?it/s]

[I 2024-04-19 14:31:30,342] A new study created in memory with name: no-name-3e48b656-3af5-4fe1-af9d-750625779caa


Fold 1 C-index: 0.6294820717131474
Fold 2 C-index: 0.7751937984496124
Fold 3 C-index: 0.6936170212765957
Fold 4 C-index: 0.7642585551330798
Fold 5 C-index: 0.759656652360515
[I 2024-04-19 14:31:30,331] Trial 0 finished with value: 0.72444161978659 and parameters: {}. Best is trial 0 with value: 0.72444161978659.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.72444161978659], datetime_start=datetime.datetime(2024, 4, 19, 14, 31, 26, 386182), datetime_complete=datetime.datetime(2024, 4, 19, 14, 31, 30, 330646), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.72444161978659


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.22218111763385023
Fold 2 IBS: 0.18214519470787685
Fold 3 IBS: 0.19943058215780185
Fold 4 IBS: 0.2249881896341226
Fold 5 IBS: 0.1707910917826678
[I 2024-04-19 14:31:30,621] Trial 0 finished with value: 0.19990723518326386 and parameters: {}. Best is trial 0 with value: 0.19990723518326386.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.19990723518326386], datetime_start=datetime.datetime(2024, 4, 19, 14, 31, 30, 393105), datetime_complete=datetime.datetime(2024, 4, 19, 14, 31, 30, 620947), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.19990723518326386


In [30]:
# Setting a dictionary to save the train results 
train_cindex = {} 
train_ibs = {} 

# Saving the values to the dictionary 
train_cindex['CoxPH'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxPH'] = np.round(study_ibs.best_value, 3)

In [31]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.724
train_ibs:  0.2


#### Test

In [32]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [33]:
# Test on MAASTRO 
cph = CoxPHSurvivalAnalysis()

cph.fit(X_new_std, y)

# Save C-index 
c_index = cph.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print('Concordance index:', c_index)

# Save IBS 
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in cph.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print('IBS score:', ibs)

CoxPHSurvivalAnalysis()

Concordance index: 0.562
IBS score: 0.29


In [34]:
# Setting a dictionary to save the test results 
test_cindex = {} 
test_ibs = {} 

In [35]:
# Saving the values to the dictionary 
test_cindex['CoxPH'] = c_index
test_ibs['CoxPH'] = ibs

### 2. CoxnetSurvivalAnalysis - Ridge 

#### Train

In [36]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

# Running to optuna for hyperparameter tuning 
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class(l1_ratio=0.0000001, 
                            fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Robust Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = RobustScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-19 14:31:30,810] A new study created in memory with name: no-name-eb912af5-aac0-47b6-bdb8-c3a56f30c091


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.5776892430278885
Fold 2 C-index: 0.7151162790697675
Fold 3 C-index: 0.5595744680851064
Fold 4 C-index: 0.7072243346007605


[I 2024-04-19 14:31:30,972] A new study created in memory with name: no-name-89d7537d-8e2e-4c60-813e-cc4671de1911


Fold 5 C-index: 0.6952789699570815
[I 2024-04-19 14:31:30,967] Trial 0 finished with value: 0.6509766589481208 and parameters: {}. Best is trial 0 with value: 0.6509766589481208.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.6509766589481208], datetime_start=datetime.datetime(2024, 4, 19, 14, 31, 30, 846174), datetime_complete=datetime.datetime(2024, 4, 19, 14, 31, 30, 966942), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.6509766589481208


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.24724709781339987
Fold 2 IBS: 0.2320398746717657
Fold 3 IBS: 0.2289818676351831
Fold 4 IBS: 0.2419747644101402
Fold 5 IBS: 0.22939558596182483
[I 2024-04-19 14:31:31,156] Trial 0 finished with value: 0.23592783809846277 and parameters: {}. Best is trial 0 with value: 0.23592783809846277.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.23592783809846277], datetime_start=datetime.datetime(2024, 4, 19, 14, 31, 31, 4197), datetime_complete=datetime.datetime(2024, 4, 19, 14, 31, 31, 156102), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.23592783809846277


In [37]:
train_cindex['CoxRidge'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxRidge'] = np.round(study_ibs.best_value, 3)

In [38]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.651
train_ibs:  0.236


#### Test

In [39]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [40]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params['l1_ratio'] = 0.0000001
    best_params['fit_baseline_model']=True
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1e-07)

test_cindex : 0.535


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1e-07)

test_ibs:  0.229


In [41]:
# Saving the values to the dictionary 
test_cindex['CoxRidge'] = c_index
test_ibs['CoxRidge'] = ibs

### 3. CoxnetSurvivalAnalysis - Lasso

#### Train

In [42]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class(l1_ratio=1, 
                            fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Robust Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = RobustScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-19 14:31:31,338] A new study created in memory with name: no-name-842b286e-fd38-4e53-86ef-9c1e1159a29f


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.6254980079681275


[I 2024-04-19 14:31:31,659] A new study created in memory with name: no-name-050974c2-c318-4528-a251-44ba711d5f78


Fold 2 C-index: 0.7751937984496124
Fold 3 C-index: 0.6936170212765957
Fold 4 C-index: 0.7642585551330798
Fold 5 C-index: 0.7639484978540773
[I 2024-04-19 14:31:31,655] Trial 0 finished with value: 0.7245031761362986 and parameters: {}. Best is trial 0 with value: 0.7245031761362986.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.7245031761362986], datetime_start=datetime.datetime(2024, 4, 19, 14, 31, 31, 386966), datetime_complete=datetime.datetime(2024, 4, 19, 14, 31, 31, 655021), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.7245031761362986


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.22253167582735317
Fold 2 IBS: 0.1820013069462648
Fold 3 IBS: 0.19910377379051175
Fold 4 IBS: 0.22395364989497227
Fold 5 IBS: 0.1685762806839025
[I 2024-04-19 14:31:32,004] Trial 0 finished with value: 0.1992333374286009 and parameters: {}. Best is trial 0 with value: 0.1992333374286009.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.1992333374286009], datetime_start=datetime.datetime(2024, 4, 19, 14, 31, 31, 692517), datetime_complete=datetime.datetime(2024, 4, 19, 14, 31, 32, 4010), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.1992333374286009


In [43]:
train_cindex['CoxLasso'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxLasso'] = np.round(study_ibs.best_value, 3)

In [44]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.725
train_ibs:  0.199


#### Test 

In [45]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [46]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params['l1_ratio'] = 1
    best_params['fit_baseline_model']=True
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1)

test_cindex : 0.561


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1)

test_ibs:  0.288


In [47]:
# Saving the values to the dictionary 
test_cindex['CoxLasso'] = c_index
test_ibs['CoxLasso'] = ibs

### 4. CoxnetSurvivalAnalysis - ElasticNet

#### Train

In [48]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        l1_ratio = trial.suggest_float("l1_ratio", 0.0001, 1)
        
        # Create and fit survival model 
        model = model_class(l1_ratio=l1_ratio, 
                           fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Robust Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = RobustScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-19 14:31:32,218] A new study created in memory with name: no-name-64d6b652-56a0-4b40-b01e-c58318386128


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.6254980079681275
Fold 2 C-index: 0.7751937984496124
Fold 3 C-index: 0.6936170212765957
Fold 4 C-index: 0.7642585551330798
Fold 5 C-index: 0.7682403433476395
[I 2024-04-19 14:31:32,499] Trial 0 finished with value: 0.725361545235011 and parameters: {'l1_ratio': 0.6964995386793018}. Best is trial 0 with value: 0.725361545235011.
Fold 1 C-index: 0.6215139442231076
Fold 2 C-index: 0.7635658914728682
Fold 3 C-index: 0.7063829787234043
Fold 4 C-index: 0.7680608365019012
Fold 5 C-index: 0.7682403433476395
[I 2024-04-19 14:31:32,822] Trial 1 finished with value: 0.7255527988537841 and parameters: {'l1_ratio': 0.28621072101688444}. Best is trial 1 with value: 0.7255527988537841.
Fold 1 C-index: 0.6254980079681275
Fold 2 C-index: 0.7635658914728682
Fold 3 C-index: 0.7063829787234043
Fold 4 C-index: 0.7680608365019012
Fold 5 C-index: 0.7639484978540773
[I 2024-04-19 14:31:33,073] Trial 2 finished with value: 0.7254912425040756 and parameters: {'l1_ratio': 0.22692876841884668}. B

Fold 1 C-index: 0.6254980079681275
Fold 2 C-index: 0.7635658914728682
Fold 3 C-index: 0.6978723404255319
Fold 4 C-index: 0.7642585551330798
Fold 5 C-index: 0.7682403433476395
[I 2024-04-19 14:31:38,658] Trial 24 finished with value: 0.7238870276694493 and parameters: {'l1_ratio': 0.4863655198534047}. Best is trial 1 with value: 0.7255527988537841.
Fold 1 C-index: 0.6215139442231076
Fold 2 C-index: 0.7635658914728682
Fold 3 C-index: 0.7063829787234043
Fold 4 C-index: 0.7680608365019012
Fold 5 C-index: 0.7682403433476395
[I 2024-04-19 14:31:38,912] Trial 25 finished with value: 0.7255527988537841 and parameters: {'l1_ratio': 0.31701825327701266}. Best is trial 1 with value: 0.7255527988537841.
Fold 1 C-index: 0.6254980079681275
Fold 2 C-index: 0.7635658914728682
Fold 3 C-index: 0.5617021276595745
Fold 4 C-index: 0.6844106463878327
Fold 5 C-index: 0.7639484978540773
[I 2024-04-19 14:31:39,105] Trial 26 finished with value: 0.6798250342684959 and parameters: {'l1_ratio': 0.0944565159885475

Fold 1 C-index: 0.6215139442231076
Fold 2 C-index: 0.7635658914728682
Fold 3 C-index: 0.7063829787234043
Fold 4 C-index: 0.7680608365019012
Fold 5 C-index: 0.7682403433476395
[I 2024-04-19 14:31:44,344] Trial 48 finished with value: 0.7255527988537841 and parameters: {'l1_ratio': 0.26268375459327914}. Best is trial 1 with value: 0.7255527988537841.
Fold 1 C-index: 0.6294820717131474
Fold 2 C-index: 0.7170542635658915
Fold 3 C-index: 0.5617021276595745
Fold 4 C-index: 0.6920152091254753
Fold 5 C-index: 0.6952789699570815
[I 2024-04-19 14:31:44,475] Trial 49 finished with value: 0.659106528404234 and parameters: {'l1_ratio': 0.04475149791307237}. Best is trial 1 with value: 0.7255527988537841.
Fold 1 C-index: 0.6254980079681275
Fold 2 C-index: 0.7635658914728682
Fold 3 C-index: 0.7063829787234043
Fold 4 C-index: 0.7680608365019012
Fold 5 C-index: 0.7682403433476395
[I 2024-04-19 14:31:44,752] Trial 50 finished with value: 0.7263496116027881 and parameters: {'l1_ratio': 0.2189074747531021

...

  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 IBS: 0.2225704262218376
Fold 2 IBS: 0.1819498270902588
Fold 3 IBS: 0.19886768747656922
Fold 4 IBS: 0.22386567170541638
Fold 5 IBS: 0.16840167511994572
[I 2024-04-19 14:32:00,256] Trial 0 finished with value: 0.19913105752280555 and parameters: {'l1_ratio': 0.6964995386793018}. Best is trial 0 with value: 0.19913105752280555.
Fold 1 IBS: 0.22256851645545656
Fold 2 IBS: 0.18183149525695486
Fold 3 IBS: 0.19831127087456324
Fold 4 IBS: 0.22374002106557006
Fold 5 IBS: 0.16804963785268703
[I 2024-04-19 14:32:00,645] Trial 1 finished with value: 0.19890018830104633 and parameters: {'l1_ratio': 0.28621072101688444}. Best is trial 1 with value: 0.19890018830104633.
Fold 1 IBS: 0.22256240629167903
Fold 2 IBS: 0.18180990729063937
Fold 3 IBS: 0.1980949387348138
Fold 4 IBS: 0.2237057863023476
Fold 5 IBS: 0.16798363636883748
[I 2024-04-19 14:32:00,978] Trial 2 finished with value: 0.19883133499766345 and parameters: {'l1_ratio': 0.22692876841884668}. Best is trial 2 with value: 0.1988313349976

Fold 2 IBS: 0.18182827943475607
Fold 3 IBS: 0.1983027056168231
Fold 4 IBS: 0.2236969058051047
Fold 5 IBS: 0.16807672058593062
[I 2024-04-19 14:32:11,275] Trial 25 finished with value: 0.19889801384863598 and parameters: {'l1_ratio': 0.30614008618835464}. Best is trial 20 with value: 0.198749327965032.
Fold 1 IBS: 0.2225581693835911
Fold 2 IBS: 0.18175084032337233
Fold 3 IBS: 0.19791958521091388
Fold 4 IBS: 0.22357324265909004
Fold 5 IBS: 0.16788657871133672
[I 2024-04-19 14:32:11,692] Trial 26 finished with value: 0.1987376832576608 and parameters: {'l1_ratio': 0.11368595762521919}. Best is trial 26 with value: 0.1987376832576608.
Fold 1 IBS: 0.22257263067044297
Fold 2 IBS: 0.18192383838742884
Fold 3 IBS: 0.19874727605699896
Fold 4 IBS: 0.22381170476765064
Fold 5 IBS: 0.16832179413382806
[I 2024-04-19 14:32:12,143] Trial 27 finished with value: 0.1990754488032699 and parameters: {'l1_ratio': 0.58416616744615}. Best is trial 26 with value: 0.1987376832576608.
Fold 1 IBS: 0.2225385559577

Fold 2 IBS: 0.18183484175067388
Fold 3 IBS: 0.19833177688925338
Fold 4 IBS: 0.22372220531719153
Fold 5 IBS: 0.168075203534567
[I 2024-04-19 14:32:24,593] Trial 50 finished with value: 0.19890878601447065 and parameters: {'l1_ratio': 0.31297686520002327}. Best is trial 26 with value: 0.1987376832576608.
Fold 1 IBS: 0.22255942575194992
Fold 2 IBS: 0.18175788943781737
Fold 3 IBS: 0.19795379708750568
Fold 4 IBS: 0.22358765093995706
Fold 5 IBS: 0.16787604959358524
[I 2024-04-19 14:32:25,234] Trial 51 finished with value: 0.19874696256216307 and parameters: {'l1_ratio': 0.1265696579057886}. Best is trial 26 with value: 0.1987376832576608.
Fold 1 IBS: 0.22258042673434433
Fold 2 IBS: 0.18175992873615382
Fold 3 IBS: 0.197961699317436
Fold 4 IBS: 0.22361263992858413
Fold 5 IBS: 0.16787018091678751
[I 2024-04-19 14:32:25,882] Trial 52 finished with value: 0.19875697512666118 and parameters: {'l1_ratio': 0.11686454792811474}. Best is trial 26 with value: 0.1987376832576608.
Fold 1 IBS: 0.222558048

Fold 2 IBS: 0.18189566613314634
Fold 3 IBS: 0.1986289263483389
Fold 4 IBS: 0.2238144063367587
Fold 5 IBS: 0.16827084120297273
[I 2024-04-19 14:32:38,181] Trial 75 finished with value: 0.19903430291539004 and parameters: {'l1_ratio': 0.523110210901512}. Best is trial 61 with value: 0.19873443518910555.
Fold 1 IBS: 0.22255792160452792
Fold 2 IBS: 0.18176901837548343
Fold 3 IBS: 0.19800787253280158
Fold 4 IBS: 0.22361774383674754
Fold 5 IBS: 0.16789995287837237
[I 2024-04-19 14:32:38,592] Trial 76 finished with value: 0.19877050184558656 and parameters: {'l1_ratio': 0.14271358093225087}. Best is trial 61 with value: 0.19873443518910555.
Fold 1 IBS: 0.2225804671366443
Fold 2 IBS: 0.18180060546395152
Fold 3 IBS: 0.1980408252218206
Fold 4 IBS: 0.22371059269718394
Fold 5 IBS: 0.16794118873714264
[I 2024-04-19 14:32:39,374] Trial 77 finished with value: 0.19881473585134862 and parameters: {'l1_ratio': 0.18670775174471713}. Best is trial 61 with value: 0.19873443518910555.
Fold 1 IBS: 0.2472380

In [49]:
train_cindex['CoxElastic'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxElastic'] = np.round(study_ibs.best_value, 3)

In [50]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.726
train_ibs:  0.199


#### Test

In [51]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [52]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    return model_class(**best_params, fit_baseline_model=True)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=0.21890747475310213)

test_cindex : 0.56


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=0.10729770639367668)

test_ibs:  0.289


In [53]:
# Saving the values to the dictionary 
test_cindex['CoxElastic'] = c_index
test_ibs['CoxElastic'] = ibs

### 5. Random Survival Forest

#### Train

In [54]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        oob_score = trial.suggest_categorical("oob_score", [True, False])
        max_samples = trial.suggest_float("max_samples", 0.1, 1.0) 
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None])
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        
        # Include warm_start for C-index optimization
        if metric == "c-index":
            warm_start = trial.suggest_categorical("warm_start", [True, False])
        else:
            warm_start = False  # Exclude warm_start for other metrics
        
        # Create and fit survival model 
        model = model_class(min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            max_leaf_nodes=max_leaf_nodes,
                            n_estimators=n_estimators, 
                            oob_score=oob_score,
                            warm_start=warm_start,
                            max_depth=max_depth,
                            max_features=max_features,
                            min_weight_fraction_leaf=min_weight_fraction_leaf, 
                            max_samples=max_samples, 
                            random_state=123)

        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])
            
            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(RandomSurvivalForest, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(RandomSurvivalForest, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-19 14:32:58,424] A new study created in memory with name: no-name-88a5c509-d094-4074-829e-215034fbbdec


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.6334661354581673
Fold 2 C-index: 0.7674418604651163
Fold 3 C-index: 0.6510638297872341
Fold 4 C-index: 0.752851711026616
Fold 5 C-index: 0.6909871244635193
[I 2024-04-19 14:33:04,901] Trial 0 finished with value: 0.6991621322401306 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 7, 'min_samples_leaf': 5, 'max_depth': 12, 'n_estimators': 360, 'oob_score': False, 'max_samples': 0.7163467647263769, 'max_features': None, 'min_weight_fraction_leaf': 0.2192861223398122, 'warm_start': False}. Best is trial 0 with value: 0.6991621322401306.
Fold 1 C-index: 0.6414342629482072
Fold 2 C-index: 0.7713178294573644
Fold 3 C-index: 0.7191489361702128
Fold 4 C-index: 0.7946768060836502
Fold 5 C-index: 0.7510729613733905
[I 2024-04-19 14:33:09,270] Trial 1 finished with value: 0.7355301592065651 and parameters: {'min_samples_split': 16, 'max_leaf_nodes': 5, 'min_samples_leaf': 4, 'max_depth': 11, 'n_estimators': 266, 'oob_score': False, 'max_samples': 0.7520097923745717, '

Fold 3 C-index: 0.7446808510638298
Fold 4 C-index: 0.8022813688212928
Fold 5 C-index: 0.759656652360515
[I 2024-04-19 14:33:57,284] Trial 15 finished with value: 0.7430557612306835 and parameters: {'min_samples_split': 9, 'max_leaf_nodes': 16, 'min_samples_leaf': 8, 'max_depth': 1, 'n_estimators': 114, 'oob_score': True, 'max_samples': 0.3692598016141903, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.17993556917544976, 'warm_start': True}. Best is trial 14 with value: 0.7726196368380054.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-19 14:33:57,651] Trial 16 finished with value: 0.5 and parameters: {'min_samples_split': 7, 'max_leaf_nodes': 16, 'min_samples_leaf': 8, 'max_depth': 1, 'n_estimators': 6, 'oob_score': True, 'max_samples': 0.36871324569404207, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.1937382585942945, 'warm_start': True}. Best is trial 14 with value: 0.7726196368380054.
Fold 1 C-index: 0.5

Fold 1 C-index: 0.6095617529880478
Fold 2 C-index: 0.8914728682170543
Fold 3 C-index: 0.9276595744680851
Fold 4 C-index: 0.8935361216730038
Fold 5 C-index: 0.8969957081545065
[I 2024-04-19 14:34:12,336] Trial 31 finished with value: 0.8438452051001395 and parameters: {'min_samples_split': 2, 'max_leaf_nodes': 8, 'min_samples_leaf': 3, 'max_depth': 5, 'n_estimators': 166, 'oob_score': True, 'max_samples': 0.8383718236388968, 'max_features': None, 'min_weight_fraction_leaf': 0.05349821135610229, 'warm_start': True}. Best is trial 30 with value: 0.8471808956568762.
Fold 1 C-index: 0.6215139442231076
Fold 2 C-index: 0.8875968992248062
Fold 3 C-index: 0.9319148936170213
Fold 4 C-index: 0.8935361216730038
Fold 5 C-index: 0.8969957081545065
[I 2024-04-19 14:34:14,070] Trial 32 finished with value: 0.8463115133784891 and parameters: {'min_samples_split': 2, 'max_leaf_nodes': 8, 'min_samples_leaf': 3, 'max_depth': 5, 'n_estimators': 165, 'oob_score': True, 'max_samples': 0.844850651135347, 'max

...

Fold 5 IBS: 0.17585992091527558
[I 2024-04-19 14:40:40,494] Trial 31 finished with value: 0.19271571153723607 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 13, 'min_samples_leaf': 2, 'max_depth': 4, 'n_estimators': 374, 'oob_score': False, 'max_samples': 0.7600957156160437, 'max_features': 'auto', 'min_weight_fraction_leaf': 0.045073432551245296}. Best is trial 30 with value: 0.19201902148306207.
Fold 1 IBS: 0.22236771464371266
Fold 2 IBS: 0.17701546684229433
Fold 3 IBS: 0.19851905347240079
Fold 4 IBS: 0.18732302760178435
Fold 5 IBS: 0.18167933628393707
[I 2024-04-19 14:40:46,595] Trial 32 finished with value: 0.19338091976882582 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 12, 'min_samples_leaf': 4, 'max_depth': 4, 'n_estimators': 394, 'oob_score': False, 'max_samples': 0.6752379846937863, 'max_features': 'auto', 'min_weight_fraction_leaf': 0.07718385932974292}. Best is trial 30 with value: 0.19201902148306207.
Fold 1 IBS: 0.2270170034083123
Fold 2 IBS: 0.17

Fold 1 IBS: 0.22670665484519142
Fold 2 IBS: 0.175184981867077
Fold 3 IBS: 0.20276219990812896
Fold 4 IBS: 0.19658107727261281
Fold 5 IBS: 0.17640448174832232
[I 2024-04-19 14:42:14,936] Trial 47 finished with value: 0.1955278791282665 and parameters: {'min_samples_split': 4, 'max_leaf_nodes': 15, 'min_samples_leaf': 1, 'max_depth': 14, 'n_estimators': 137, 'oob_score': True, 'max_samples': 0.7533756239183884, 'max_features': 'auto', 'min_weight_fraction_leaf': 0.05433383705425634}. Best is trial 41 with value: 0.19046423845342159.
Fold 1 IBS: 0.21833683794653208
Fold 2 IBS: 0.16845072186321247
Fold 3 IBS: 0.20168040343516955
Fold 4 IBS: 0.1937674426294509
Fold 5 IBS: 0.19051837676249267
[I 2024-04-19 14:42:16,886] Trial 48 finished with value: 0.19455075652737155 and parameters: {'min_samples_split': 6, 'max_leaf_nodes': 11, 'min_samples_leaf': 3, 'max_depth': 4, 'n_estimators': 82, 'oob_score': False, 'max_samples': 0.7075209195984526, 'max_features': 'auto', 'min_weight_fraction_leaf

Fold 5 IBS: 0.16839194066878888
[I 2024-04-19 14:43:21,761] Trial 62 finished with value: 0.1913839096782827 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 11, 'min_samples_leaf': 4, 'max_depth': 6, 'n_estimators': 298, 'oob_score': False, 'max_samples': 0.8075940187858324, 'max_features': 'sqrt', 'min_weight_fraction_leaf': 0.0028869935676452026}. Best is trial 49 with value: 0.19023957913930975.
Fold 1 IBS: 0.22802304509837604
Fold 2 IBS: 0.17093887708958988
Fold 3 IBS: 0.19997810028764312
Fold 4 IBS: 0.19416891669551373
Fold 5 IBS: 0.16864914160971897
[I 2024-04-19 14:43:27,695] Trial 63 finished with value: 0.19235161615616833 and parameters: {'min_samples_split': 6, 'max_leaf_nodes': 11, 'min_samples_leaf': 4, 'max_depth': 7, 'n_estimators': 305, 'oob_score': False, 'max_samples': 0.899801190989709, 'max_features': 'sqrt', 'min_weight_fraction_leaf': 0.0075083578814244606}. Best is trial 49 with value: 0.19023957913930975.
Fold 1 IBS: 0.22706249643509485
Fold 2 IBS: 0.

Fold 1 IBS: 0.22661384896598818
Fold 2 IBS: 0.20671937958697834
Fold 3 IBS: 0.22631312111115084
Fold 4 IBS: 0.21497952798665046
Fold 5 IBS: 0.21282749449450544
[I 2024-04-19 14:44:45,803] Trial 78 finished with value: 0.21749067442905465 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 9, 'min_samples_leaf': 7, 'max_depth': 6, 'n_estimators': 289, 'oob_score': False, 'max_samples': 0.9147835512625551, 'max_features': 'sqrt', 'min_weight_fraction_leaf': 0.43785214928176214}. Best is trial 49 with value: 0.19023957913930975.
Fold 1 IBS: 0.23052477127318172
Fold 2 IBS: 0.16982397129659127
Fold 3 IBS: 0.197745679074635
Fold 4 IBS: 0.19675496948041768
Fold 5 IBS: 0.1770213286573074
[I 2024-04-19 14:44:51,710] Trial 79 finished with value: 0.1943741439564266 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 12, 'min_samples_leaf': 6, 'max_depth': 7, 'n_estimators': 275, 'oob_score': False, 'max_samples': 0.9974228392350986, 'max_features': 'sqrt', 'min_weight_fraction_leaf

Fold 5 IBS: 0.16901501941163646
[I 2024-04-19 14:45:55,332] Trial 93 finished with value: 0.19108673519318098 and parameters: {'min_samples_split': 8, 'max_leaf_nodes': 9, 'min_samples_leaf': 2, 'max_depth': 9, 'n_estimators': 179, 'oob_score': False, 'max_samples': 0.8387093701884653, 'max_features': 'sqrt', 'min_weight_fraction_leaf': 0.030817946786640082}. Best is trial 49 with value: 0.19023957913930975.
Fold 1 IBS: 0.22184856220737365
Fold 2 IBS: 0.17312992964385657
Fold 3 IBS: 0.1948930255496256
Fold 4 IBS: 0.1925411854591036
Fold 5 IBS: 0.1730631403095586
[I 2024-04-19 14:45:59,252] Trial 94 finished with value: 0.1910951686339036 and parameters: {'min_samples_split': 8, 'max_leaf_nodes': 9, 'min_samples_leaf': 2, 'max_depth': 9, 'n_estimators': 182, 'oob_score': False, 'max_samples': 0.688472014648766, 'max_features': 'sqrt', 'min_weight_fraction_leaf': 0.03451598429298541}. Best is trial 49 with value: 0.19023957913930975.
Fold 1 IBS: 0.24047052597810953
Fold 2 IBS: 0.17250441

In [55]:
train_cindex['Randomsurvivalforest'] = np.round(study_cindex.best_value, 3)
train_ibs['Randomsurvivalforest'] = np.round(study_ibs.best_value, 3)

In [56]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.866
train_ibs:  0.19


#### Test

In [57]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))
    
y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [58]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(RandomSurvivalForest, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex: ", c_index)

# Set the best model 
best_model_ibs = create_best_model(RandomSurvivalForest, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

RandomSurvivalForest(max_depth=10, max_leaf_nodes=14,
                     max_samples=0.8746458616251406, min_samples_leaf=2,
                     min_weight_fraction_leaf=0.030922977401692878,
                     n_estimators=113, random_state=123, warm_start=True)

test_cindex:  0.546


RandomSurvivalForest(max_depth=8, max_leaf_nodes=17,
                     max_samples=0.7890788621794695, min_samples_leaf=2,
                     min_samples_split=2,
                     min_weight_fraction_leaf=0.04458702793409107,
                     n_estimators=215, random_state=123)

test_ibs:  0.263


In [59]:
# Saving the values to the dictionary 
test_cindex['Randomsurvivalforest'] = c_index
test_ibs['Randomsurvivalforest'] = ibs

### 6. ExtraSurvivalTrees

#### Train

In [60]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

In [61]:
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters 
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        oob_score = trial.suggest_categorical("oob_score", [True, False])
        warm_start = trial.suggest_categorical("warm_start", [True, False])
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None, 0.1, 1])
        max_samples = trial.suggest_float("max_samples", 0.1, 1.0) 
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        
        # Include warm_start for C-index optimization
        if metric == "c-index":
            warm_start = trial.suggest_categorical("warm_start", [True, False])
        else:
            warm_start = False  # Exclude warm_start for other metrics

        
        # Create and fit survival model 
        model = model_class(min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            max_leaf_nodes=max_leaf_nodes,
                            n_estimators=n_estimators, 
                            oob_score=oob_score, 
                            max_features=max_features, 
                            warm_start=warm_start, 
                            max_samples=max_samples,
                            min_weight_fraction_leaf=min_weight_fraction_leaf, 
                            max_depth=max_depth, 
                            random_state=123) 
        
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(ExtraSurvivalTrees, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(ExtraSurvivalTrees, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-19 14:46:25,083] A new study created in memory with name: no-name-697d832d-732d-4a96-af7c-bdfd83d034ac


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.6573705179282868
Fold 2 C-index: 0.7364341085271318
Fold 3 C-index: 0.8553191489361702
Fold 4 C-index: 0.7984790874524715
Fold 5 C-index: 0.7639484978540773
[I 2024-04-19 14:46:27,002] Trial 0 finished with value: 0.7623102721396274 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 7, 'min_samples_leaf': 5, 'max_depth': 12, 'n_estimators': 360, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.7641958651588321, 'min_weight_fraction_leaf': 0.09124586522674999}. Best is trial 0 with value: 0.7623102721396274.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-19 14:46:30,644] Trial 1 finished with value: 0.5 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 12, 'min_samples_leaf': 11, 'max_depth': 13, 'n_estimators': 425, 'oob_score': True, 'warm_start': True, 'max_features': None, 'max_samples': 0.4877764869966794, 'min_weight_fraction_leaf': 0.2468425488251531

Fold 1 C-index: 0.6115537848605578
Fold 2 C-index: 0.7131782945736435
Fold 3 C-index: 0.7872340425531915
Fold 4 C-index: 0.7281368821292775
Fold 5 C-index: 0.6759656652360515
[I 2024-04-19 14:47:12,573] Trial 16 finished with value: 0.7032137338705444 and parameters: {'min_samples_split': 20, 'max_leaf_nodes': 17, 'min_samples_leaf': 5, 'max_depth': 20, 'n_estimators': 499, 'oob_score': False, 'warm_start': True, 'max_features': 'auto', 'max_samples': 0.8750346354626457, 'min_weight_fraction_leaf': 0.4093818399278544}. Best is trial 12 with value: 0.7867972827001483.
Fold 1 C-index: 0.6454183266932271
Fold 2 C-index: 0.7325581395348837
Fold 3 C-index: 0.825531914893617
Fold 4 C-index: 0.7718631178707225
Fold 5 C-index: 0.6824034334763949
[I 2024-04-19 14:47:14,138] Trial 17 finished with value: 0.731554986493769 and parameters: {'min_samples_split': 17, 'max_leaf_nodes': 10, 'min_samples_leaf': 8, 'max_depth': 12, 'n_estimators': 318, 'oob_score': False, 'warm_start': True, 'max_featur

Fold 1 C-index: 0.649402390438247
Fold 2 C-index: 0.7325581395348837
Fold 3 C-index: 0.8382978723404255
Fold 4 C-index: 0.7984790874524715
Fold 5 C-index: 0.8068669527896996
[I 2024-04-19 14:47:46,174] Trial 31 finished with value: 0.7651208885111453 and parameters: {'min_samples_split': 9, 'max_leaf_nodes': 5, 'min_samples_leaf': 1, 'max_depth': 5, 'n_estimators': 400, 'oob_score': False, 'warm_start': True, 'max_features': 1, 'max_samples': 0.6357896182918609, 'min_weight_fraction_leaf': 0.043457520259011145}. Best is trial 23 with value: 0.7937218312264874.
Fold 1 C-index: 0.6414342629482072
Fold 2 C-index: 0.7286821705426356
Fold 3 C-index: 0.8340425531914893
Fold 4 C-index: 0.7870722433460076
Fold 5 C-index: 0.7553648068669528
[I 2024-04-19 14:47:48,370] Trial 32 finished with value: 0.7493192073790584 and parameters: {'min_samples_split': 13, 'max_leaf_nodes': 6, 'min_samples_leaf': 1, 'max_depth': 4, 'n_estimators': 400, 'oob_score': False, 'warm_start': True, 'max_features': 1,

...

Fold 1 IBS: 0.2233579319393271
Fold 2 IBS: 0.20096408163499427
Fold 3 IBS: 0.19049881866793233
Fold 4 IBS: 0.20774755146340546
Fold 5 IBS: 0.19123419554925844
[I 2024-04-19 14:53:39,189] Trial 30 finished with value: 0.20276051585098354 and parameters: {'min_samples_split': 4, 'max_leaf_nodes': 3, 'min_samples_leaf': 4, 'max_depth': 12, 'n_estimators': 355, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.7262614295600427, 'min_weight_fraction_leaf': 0.020774312173092817}. Best is trial 23 with value: 0.19459187877865486.
Fold 1 IBS: 0.2179837541713329
Fold 2 IBS: 0.19996868148399857
Fold 3 IBS: 0.18950194352113112
Fold 4 IBS: 0.20402491486791338
Fold 5 IBS: 0.18472430994108335
[I 2024-04-19 14:53:47,229] Trial 31 finished with value: 0.19924072079709187 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 8, 'min_samples_leaf': 5, 'max_depth': 12, 'n_estimators': 409, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 

Fold 1 IBS: 0.22018361682359297
Fold 2 IBS: 0.19767576801393197
Fold 3 IBS: 0.19421270363305465
Fold 4 IBS: 0.20406059613721392
Fold 5 IBS: 0.1854331539519078
[I 2024-04-19 14:55:56,445] Trial 45 finished with value: 0.20031316771194024 and parameters: {'min_samples_split': 19, 'max_leaf_nodes': 12, 'min_samples_leaf': 2, 'max_depth': 14, 'n_estimators': 477, 'oob_score': True, 'warm_start': False, 'max_features': None, 'max_samples': 0.8420791678401572, 'min_weight_fraction_leaf': 0.1114478053387663}. Best is trial 34 with value: 0.1915510180929927.
Fold 1 IBS: 0.22677495195617972
Fold 2 IBS: 0.20938982706068335
Fold 3 IBS: 0.20301008325030304
Fold 4 IBS: 0.21538736174154596
Fold 5 IBS: 0.2016448459853074
[I 2024-04-19 14:56:06,190] Trial 46 finished with value: 0.21124141399880392 and parameters: {'min_samples_split': 18, 'max_leaf_nodes': 10, 'min_samples_leaf': 1, 'max_depth': 15, 'n_estimators': 432, 'oob_score': True, 'warm_start': False, 'max_features': 0.1, 'max_samples': 0.945

Fold 1 IBS: 0.21476513837056027
Fold 2 IBS: 0.18143050987640613
Fold 3 IBS: 0.1908356407571883
Fold 4 IBS: 0.19427180424062881
Fold 5 IBS: 0.16683442835564766
[I 2024-04-19 14:58:24,012] Trial 60 finished with value: 0.18962750432008624 and parameters: {'min_samples_split': 13, 'max_leaf_nodes': 12, 'min_samples_leaf': 3, 'max_depth': 11, 'n_estimators': 439, 'oob_score': True, 'warm_start': False, 'max_features': None, 'max_samples': 0.8952499671372233, 'min_weight_fraction_leaf': 0.036398817545426995}. Best is trial 60 with value: 0.18962750432008624.
Fold 1 IBS: 0.21627361369698
Fold 2 IBS: 0.1817225137067132
Fold 3 IBS: 0.19073393689024337
Fold 4 IBS: 0.19494098781318964
Fold 5 IBS: 0.16750297353174665
[I 2024-04-19 14:58:36,491] Trial 61 finished with value: 0.19023480512777455 and parameters: {'min_samples_split': 13, 'max_leaf_nodes': 12, 'min_samples_leaf': 3, 'max_depth': 11, 'n_estimators': 438, 'oob_score': True, 'warm_start': False, 'max_features': None, 'max_samples': 0.87

Fold 1 IBS: 0.21657675292627207
Fold 2 IBS: 0.18996174454319362
Fold 3 IBS: 0.18393478555155848
Fold 4 IBS: 0.19722264590644567
Fold 5 IBS: 0.17340727825186003
[I 2024-04-19 15:00:41,356] Trial 75 finished with value: 0.19222064143586598 and parameters: {'min_samples_split': 14, 'max_leaf_nodes': 15, 'min_samples_leaf': 3, 'max_depth': 8, 'n_estimators': 391, 'oob_score': True, 'warm_start': False, 'max_features': 'auto', 'max_samples': 0.9086795540810024, 'min_weight_fraction_leaf': 0.03284410208925504}. Best is trial 63 with value: 0.1883299130608305.
Fold 1 IBS: 0.21374362171322053
Fold 2 IBS: 0.1809837734102344
Fold 3 IBS: 0.1928804193379134
Fold 4 IBS: 0.19389862294434004
Fold 5 IBS: 0.16641872761123616
[I 2024-04-19 15:00:51,911] Trial 76 finished with value: 0.1895850330033889 and parameters: {'min_samples_split': 10, 'max_leaf_nodes': 13, 'min_samples_leaf': 4, 'max_depth': 11, 'n_estimators': 417, 'oob_score': True, 'warm_start': False, 'max_features': None, 'max_samples': 0.8

Fold 1 IBS: 0.21649654230068696
Fold 2 IBS: 0.1994536066038553
Fold 3 IBS: 0.1910443715333566
Fold 4 IBS: 0.20311940988854024
Fold 5 IBS: 0.18412454653280888
[I 2024-04-19 15:02:46,604] Trial 90 finished with value: 0.1988476953718496 and parameters: {'min_samples_split': 10, 'max_leaf_nodes': 17, 'min_samples_leaf': 6, 'max_depth': 9, 'n_estimators': 305, 'oob_score': True, 'warm_start': False, 'max_features': 'auto', 'max_samples': 0.9531106840561406, 'min_weight_fraction_leaf': 0.09026278115621361}. Best is trial 63 with value: 0.1883299130608305.
Fold 1 IBS: 0.21417020956953955
Fold 2 IBS: 0.18221070479947724
Fold 3 IBS: 0.1876123093586995
Fold 4 IBS: 0.1945453292939215
Fold 5 IBS: 0.1658337819099612
[I 2024-04-19 15:03:00,896] Trial 91 finished with value: 0.1888744669863198 and parameters: {'min_samples_split': 11, 'max_leaf_nodes': 13, 'min_samples_leaf': 3, 'max_depth': 5, 'n_estimators': 390, 'oob_score': True, 'warm_start': False, 'max_features': None, 'max_samples': 0.915582

In [62]:
train_cindex['ExtraSurvivalTrees'] = np.round(study_cindex.best_value, 3)
train_ibs['ExtraSurvivalTrees'] = np.round(study_ibs.best_value, 3)

In [63]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.826
train_ibs:  0.188


#### Test

In [64]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [65]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(ExtraSurvivalTrees, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(ExtraSurvivalTrees, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

ExtraSurvivalTrees(max_depth=6, max_leaf_nodes=18,
                   max_samples=0.8161032428574773, min_samples_leaf=2,
                   min_samples_split=11,
                   min_weight_fraction_leaf=0.001106761767719347,
                   n_estimators=105, oob_score=True, random_state=123,
                   warm_start=True)

C-index score: 0.607


ExtraSurvivalTrees(max_depth=10, max_features=None, max_leaf_nodes=12,
                   max_samples=0.8664901703822724, min_samples_split=13,
                   min_weight_fraction_leaf=0.0002786019075411844,
                   n_estimators=373, oob_score=True, random_state=123)

IBS: 0.252


In [66]:
# Saving the values to the dictionary 
test_cindex['ExtraSurvivalTrees'] = c_index
test_ibs['ExtraSurvivalTrees'] = ibs

### 7. GradientBoostingSurvivalAnalysis

#### Train

In [67]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

# Running to optuna for hyperparameter tuning
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        subsample = trial.suggest_float("subsample", 0.1, 1)
        learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1)
        dropout_rate = trial.suggest_float("dropout_rate", 0.1, 1)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        criterion = trial.suggest_categorical('criterion', ['friedman_mse', 'squared_error'])
        ccp_alpha = trial.suggest_float("ccp_alpha", 0.0, 10)
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None, 0.1, 1])
        min_impurity_decrease = trial.suggest_loguniform('min_impurity_decrease', 1e-7, 1e-1)
        validation_fraction = trial.suggest_float("validation_fraction", 0.0, 1.0)
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        
        # Create and fit survival model 
        model = model_class(subsample=subsample,
                            learning_rate=learning_rate,
                            dropout_rate=dropout_rate,
                            n_estimators=n_estimators,
                            ccp_alpha=ccp_alpha, 
                            criterion=criterion,
                            min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            min_weight_fraction_leaf=min_weight_fraction_leaf,
                            max_depth=max_depth,
                            max_features=max_features,
                            max_leaf_nodes=max_leaf_nodes, 
                            min_impurity_decrease=min_impurity_decrease,
                            validation_fraction=validation_fraction, 
                            random_state=123)
        
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(GradientBoostingSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(GradientBoostingSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-19 15:04:16,943] A new study created in memory with name: no-name-c90e7784-dca9-4af7-8374-a11e65d20595


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-19 15:04:56,411] Trial 0 finished with value: 0.5 and parameters: {'subsample': 0.7268222670380755, 'learning_rate': 0.02932779416008757, 'dropout_rate': 0.3041663082077828, 'n_estimators': 276, 'criterion': 'friedman_mse', 'ccp_alpha': 9.807641983846155, 'min_weight_fraction_leaf': 0.34241486929243165, 'max_features': None, 'min_impurity_decrease': 2.4449249473284515e-05, 'validation_fraction': 0.7379954057320357, 'min_samples_split': 5, 'max_leaf_nodes': 5, 'min_samples_leaf': 11, 'max_depth': 11}. Best is trial 0 with value: 0.5.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-19 15:05:19,314] Trial 1 finished with value: 0.5 and parameters: {'subsample': 0.6709608626961889, 'learning_rate': 0.08509374761370117, 'dropout_rate': 0.7520097923745717, 'n_estimators': 306, 'criterion': 'friedman_mse', 'ccp_alpha': 3.

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-19 15:16:32,720] Trial 13 finished with value: 0.5 and parameters: {'subsample': 0.8386796524426539, 'learning_rate': 0.046734492485875676, 'dropout_rate': 0.4821375662037144, 'n_estimators': 402, 'criterion': 'squared_error', 'ccp_alpha': 1.5696007313501796, 'min_weight_fraction_leaf': 0.18684147934268416, 'max_features': 'log2', 'min_impurity_decrease': 1.5044881127471587e-06, 'validation_fraction': 0.8166356053932342, 'min_samples_split': 16, 'max_leaf_nodes': 19, 'min_samples_leaf': 16, 'max_depth': 4}. Best is trial 9 with value: 0.7053891693443892.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-19 15:17:51,651] Trial 14 finished with value: 0.5 and parameters: {'subsample': 0.33235389014851724, 'learning_rate': 0.04522573411670834, 'dropout_rate': 0.2712811374536856, 'n_estimators': 405, 'criterion': 'square

Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-19 15:32:57,324] Trial 25 finished with value: 0.5 and parameters: {'subsample': 0.8502968404151126, 'learning_rate': 0.024443951259730985, 'dropout_rate': 0.2675273969344081, 'n_estimators': 441, 'criterion': 'squared_error', 'ccp_alpha': 2.0753749717266823, 'min_weight_fraction_leaf': 0.3503497788125578, 'max_features': 'auto', 'min_impurity_decrease': 7.237153572123947e-07, 'validation_fraction': 0.41222573804914475, 'min_samples_split': 16, 'max_leaf_nodes': 13, 'min_samples_leaf': 12, 'max_depth': 3}. Best is trial 9 with value: 0.7053891693443892.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-19 15:33:35,729] Trial 26 finished with value: 0.5 and parameters: {'subsample': 0.7670085127536703, 'learning_rate': 0.011828778593594373, 'dropout_rate': 0.4300954216773497, 'n_estimators': 330, 'criterion': 'friedman_mse', 'ccp_alpha':

...

Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.22939559304809248
[I 2024-04-19 17:07:10,598] Trial 55 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.918589848048704, 'learning_rate': 0.023646082998228058, 'dropout_rate': 0.1366452847323028, 'n_estimators': 388, 'criterion': 'squared_error', 'ccp_alpha': 1.3381569877935875, 'min_weight_fraction_leaf': 0.18967222528443176, 'max_features': 'auto', 'min_impurity_decrease': 0.008146563872249941, 'validation_fraction': 0.8868940629916056, 'min_samples_split': 15, 'max_leaf_nodes': 15, 'min_samples_leaf': 19, 'max_depth': 18}. Best is trial 22 with value: 0.23482204102612672.
Fold 1 IBS: 0.24724710044658998
Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977708
Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.2293955930480925
[I 2024-04-19 17:07:56,552] Trial 56 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.7513175598857118, 'learning_rate': 0.09850922090204048, 'dropout_rate': 0.35760

Fold 4 IBS: 0.24197477145927115
Fold 5 IBS: 0.2293955930480925
[I 2024-04-19 17:15:23,529] Trial 66 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.8952643616873973, 'learning_rate': 0.05359198006915804, 'dropout_rate': 0.6509686174553241, 'n_estimators': 500, 'criterion': 'squared_error', 'ccp_alpha': 0.713342732410399, 'min_weight_fraction_leaf': 0.037327349410482574, 'max_features': 'auto', 'min_impurity_decrease': 2.2946753237767036e-06, 'validation_fraction': 0.8670144195682054, 'min_samples_split': 9, 'max_leaf_nodes': 20, 'min_samples_leaf': 14, 'max_depth': 18}. Best is trial 22 with value: 0.23482204102612672.
Fold 1 IBS: 0.24724710044658998
Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.24197477145927118
Fold 5 IBS: 0.2293955930480925
[I 2024-04-19 17:15:57,212] Trial 67 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.9559398584951578, 'learning_rate': 0.001312025546225645, 'dropout_rate': 0.8046

Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.22939559304809248
[I 2024-04-19 17:37:05,948] Trial 77 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.9040601608260395, 'learning_rate': 0.008028762030775153, 'dropout_rate': 0.1661055390191164, 'n_estimators': 412, 'criterion': 'squared_error', 'ccp_alpha': 0.5920167940309409, 'min_weight_fraction_leaf': 0.20760648939509768, 'max_features': 1, 'min_impurity_decrease': 1.2858411523836384e-06, 'validation_fraction': 0.7519570151291436, 'min_samples_split': 3, 'max_leaf_nodes': 19, 'min_samples_leaf': 8, 'max_depth': 5}. Best is trial 22 with value: 0.23482204102612672.
Fold 1 IBS: 0.24724710044658998
Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.24197477145927118
Fold 5 IBS: 0.2293955930480925
[I 2024-04-19 17:38:32,627] Trial 78 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.9412481314247185, 'learning_rate': 0.003888190949209832, 'dropout_rate': 0.625165508

Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.2293955930480925
[I 2024-04-19 19:15:22,642] Trial 88 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.7762725560789899, 'learning_rate': 0.0030671608519508686, 'dropout_rate': 0.2500967923284591, 'n_estimators': 479, 'criterion': 'squared_error', 'ccp_alpha': 1.2742275973203492, 'min_weight_fraction_leaf': 0.27772784044341225, 'max_features': 'auto', 'min_impurity_decrease': 0.0011701047770450368, 'validation_fraction': 0.9552938036430088, 'min_samples_split': 13, 'max_leaf_nodes': 19, 'min_samples_leaf': 17, 'max_depth': 2}. Best is trial 85 with value: 0.23331347435775504.
Fold 1 IBS: 0.24724710044658998
Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.2293955930480925
[I 2024-04-19 19:18:45,927] Trial 89 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.9240076064991638, 'learning_rate': 0.036372907201929795, 'dropout_rate': 0.166

Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.24197477145927115
Fold 5 IBS: 0.22939559304809248
[I 2024-04-19 19:48:46,496] Trial 99 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.9084360842440081, 'learning_rate': 0.06808689488183181, 'dropout_rate': 0.25443633448028735, 'n_estimators': 437, 'criterion': 'squared_error', 'ccp_alpha': 0.5386043790221791, 'min_weight_fraction_leaf': 0.03403879613376609, 'max_features': 'auto', 'min_impurity_decrease': 8.69596280226011e-07, 'validation_fraction': 0.9221483446288596, 'min_samples_split': 20, 'max_leaf_nodes': 12, 'min_samples_leaf': 16, 'max_depth': 2}. Best is trial 85 with value: 0.23331347435775504.


* Best trial for IBS: 
 FrozenTrial(number=85, state=TrialState.COMPLETE, values=[0.23331347435775504], datetime_start=datetime.datetime(2024, 4, 19, 19, 3, 6, 502810), datetime_complete=datetime.datetime(2024, 4, 19, 19, 7, 3, 885478), params={'subsample': 0.9481727373988897, 'learning_rate': 0.02022975259096714

In [68]:
train_cindex['GradientBoosting'] = np.round(study_cindex.best_value, 3)
train_ibs['GradientBoosting'] = np.round(study_ibs.best_value, 3)

In [69]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.759
train_ibs:  0.233


#### Test

In [70]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [71]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(GradientBoostingSurvivalAnalysis, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(GradientBoostingSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

GradientBoostingSurvivalAnalysis(ccp_alpha=0.008821356078039954,
                                 criterion='squared_error',
                                 dropout_rate=0.5604101432191428,
                                 learning_rate=0.05159117661647313, max_depth=4,
                                 max_features=0.1, max_leaf_nodes=18,
                                 min_impurity_decrease=9.803482393630746e-05,
                                 min_samples_leaf=7, min_samples_split=8,
                                 min_weight_fraction_leaf=0.26023745967507866,
                                 n_estimators=350, random_state=123,
                                 subsample=0.9197287558651637,
                                 validation_fraction=0.8247616419720114)

C-index score: 0.586


GradientBoostingSurvivalAnalysis(ccp_alpha=0.01138981446692314,
                                 criterion='squared_error',
                                 dropout_rate=0.2479619862207481,
                                 learning_rate=0.02022975259096714, max_depth=8,
                                 max_features='auto', max_leaf_nodes=20,
                                 min_impurity_decrease=9.61920586779085e-07,
                                 min_samples_leaf=18, min_samples_split=14,
                                 min_weight_fraction_leaf=0.026767353450782638,
                                 n_estimators=438, random_state=123,
                                 subsample=0.9481727373988897,
                                 validation_fraction=0.973754058089352)

IBS: 0.228


In [72]:
# Saving the values to the dictionary 
test_cindex['GradientBoosting'] = c_index
test_ibs['GradientBoosting'] = ibs

### 8. ComponentwiseGradientBoostingSurvivalAnalysis

#### Train

In [73]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

In [74]:
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        subsample = trial.suggest_float("subsample", 0.1, 1)
        dropout_rate = trial.suggest_float("dropout_rate", 0.1, 1)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1)
        
        # Create and fit survival model 
        model = model_class(subsample=subsample,
                            dropout_rate=dropout_rate,
                            n_estimators=n_estimators,
                            learning_rate=learning_rate,
                            random_state=123)
                
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Robust Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = RobustScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective


# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(ComponentwiseGradientBoostingSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best hyperparameters for C-index: \n", study_cindex.best_params)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(ComponentwiseGradientBoostingSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best hyperparameters for IBS: \n", study_ibs.best_params)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-19 19:49:38,128] A new study created in memory with name: no-name-c01abfa8-52ca-4eb5-a62f-98a2c4801f09


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5657370517928287
Fold 2 C-index: 0.6976744186046512
Fold 3 C-index: 0.5617021276595745
Fold 4 C-index: 0.6387832699619772
Fold 5 C-index: 0.6523605150214592
[I 2024-04-19 19:49:41,151] Trial 0 finished with value: 0.6232514766080982 and parameters: {'subsample': 0.7268222670380755, 'dropout_rate': 0.3575254014553415, 'n_estimators': 114, 'learning_rate': 0.05558016213920623}. Best is trial 0 with value: 0.6232514766080982.
Fold 1 C-index: 0.5657370517928287
Fold 2 C-index: 0.6976744186046512
Fold 3 C-index: 0.5617021276595745
Fold 4 C-index: 0.6387832699619772
Fold 5 C-index: 0.6523605150214592
[I 2024-04-19 19:50:04,984] Trial 1 finished with value: 0.6232514766080982 and parameters: {'subsample': 0.7475220728070068, 'dropout_rate': 0.4807958141120149, 'n_estimators': 491, 'learning_rate': 0.06879814411990147}. Best is trial 0 with value: 0.6232514766080982.
Fold 1 C-index: 0.5617529880478087
Fold 2 C-index: 0.7015503875968992
Fold 3 C-index: 0.5617021276595745
Fold 

Fold 1 C-index: 0.5856573705179283
Fold 2 C-index: 0.7209302325581395
Fold 3 C-index: 0.5574468085106383
Fold 4 C-index: 0.6920152091254753
Fold 5 C-index: 0.6824034334763949
[I 2024-04-19 19:53:23,411] Trial 19 finished with value: 0.6476906108377152 and parameters: {'subsample': 0.10013306718236009, 'dropout_rate': 0.7254192287154788, 'n_estimators': 117, 'learning_rate': 0.09614402133777997}. Best is trial 11 with value: 0.6529201065158555.
Fold 1 C-index: 0.5896414342629482
Fold 2 C-index: 0.7093023255813954
Fold 3 C-index: 0.5574468085106383
Fold 4 C-index: 0.6653992395437263
Fold 5 C-index: 0.6781115879828327
[I 2024-04-19 19:53:50,324] Trial 20 finished with value: 0.6399802791763081 and parameters: {'subsample': 0.2630057481431337, 'dropout_rate': 0.18040218016888274, 'n_estimators': 423, 'learning_rate': 0.07788582119853761}. Best is trial 11 with value: 0.6529201065158555.
Fold 1 C-index: 0.5816733067729084
Fold 2 C-index: 0.7325581395348837
Fold 3 C-index: 0.5914893617021276

Fold 1 C-index: 0.5737051792828686
Fold 2 C-index: 0.7015503875968992
Fold 3 C-index: 0.5617021276595745
Fold 4 C-index: 0.6577946768060836
Fold 5 C-index: 0.6695278969957081
[I 2024-04-19 19:54:52,904] Trial 38 finished with value: 0.6328560536682268 and parameters: {'subsample': 0.29705265581319856, 'dropout_rate': 0.14460259719192237, 'n_estimators': 30, 'learning_rate': 0.03100730936388326}. Best is trial 27 with value: 0.6672323922890245.
Fold 1 C-index: 0.5776892430278885
Fold 2 C-index: 0.7015503875968992
Fold 3 C-index: 0.5574468085106383
Fold 4 C-index: 0.6653992395437263
Fold 5 C-index: 0.6824034334763949
[I 2024-04-19 19:55:00,766] Trial 39 finished with value: 0.6368978224311095 and parameters: {'subsample': 0.22884368952376716, 'dropout_rate': 0.4678780957024635, 'n_estimators': 221, 'learning_rate': 0.04210354514561851}. Best is trial 27 with value: 0.6672323922890245.
Fold 1 C-index: 0.5776892430278885
Fold 2 C-index: 0.6976744186046512
Fold 3 C-index: 0.5617021276595745

...

Fold 3 IBS: 0.27495840642227154
Fold 4 IBS: 0.2403770465027076
Fold 5 IBS: 0.22389919673997055
[I 2024-04-19 20:11:09,715] Trial 19 finished with value: 0.24208951118640676 and parameters: {'subsample': 0.3892284838807411, 'dropout_rate': 0.5354432466556798, 'n_estimators': 131, 'learning_rate': 0.023294697221931306}. Best is trial 17 with value: 0.2265683943607694.
Fold 1 IBS: 0.24532728078212304
Fold 2 IBS: 0.21338053568843193
Fold 3 IBS: 0.23280583204014674
Fold 4 IBS: 0.23050963160208696
Fold 5 IBS: 0.21229213294998353
[I 2024-04-19 20:11:11,593] Trial 20 finished with value: 0.22686308261255445 and parameters: {'subsample': 0.6210873354088753, 'dropout_rate': 0.7933084651006226, 'n_estimators': 66, 'learning_rate': 0.013007963411749002}. Best is trial 17 with value: 0.2265683943607694.
Fold 1 IBS: 0.24480948267779246
Fold 2 IBS: 0.217894217650721
Fold 3 IBS: 0.23043919128917217
Fold 4 IBS: 0.23275995527788665
Fold 5 IBS: 0.215372968667489
[I 2024-04-19 20:11:13,203] Trial 21 finis

Fold 4 IBS: 0.23376225132374828
Fold 5 IBS: 0.21475575962300342
[I 2024-04-19 20:12:27,455] Trial 38 finished with value: 0.23328897274995045 and parameters: {'subsample': 0.605229739689187, 'dropout_rate': 0.13272164755980653, 'n_estimators': 176, 'learning_rate': 0.01283698591663533}. Best is trial 17 with value: 0.2265683943607694.
Fold 1 IBS: 0.29108106719697524
Fold 2 IBS: 0.21415810554809628
Fold 3 IBS: 0.2917820507930866
Fold 4 IBS: 0.2588982634047282
Fold 5 IBS: 0.24407574900323586
[I 2024-04-19 20:12:29,111] Trial 39 finished with value: 0.25999904718922445 and parameters: {'subsample': 0.6970162917669863, 'dropout_rate': 0.48841341315505027, 'n_estimators': 59, 'learning_rate': 0.06892741183938003}. Best is trial 17 with value: 0.2265683943607694.
Fold 1 IBS: 0.2925659515921918
Fold 2 IBS: 0.2140746731129848
Fold 3 IBS: 0.2919432513336724
Fold 4 IBS: 0.257477121073266
Fold 5 IBS: 0.24657769529804316
[I 2024-04-19 20:12:31,727] Trial 40 finished with value: 0.2605277384820316 

Fold 5 IBS: 0.27450442032616973
[I 2024-04-19 20:13:13,392] Trial 57 finished with value: 0.29085967593170914 and parameters: {'subsample': 0.763955514973389, 'dropout_rate': 0.9964559295946985, 'n_estimators': 282, 'learning_rate': 0.02203371126489022}. Best is trial 56 with value: 0.22653496264114223.
Fold 1 IBS: 0.2514876544143989
Fold 2 IBS: 0.20473165445913052
Fold 3 IBS: 0.2442786298594191
Fold 4 IBS: 0.2292932477615486
Fold 5 IBS: 0.2122783863810015
[I 2024-04-19 20:13:14,919] Trial 58 finished with value: 0.2284139145750997 and parameters: {'subsample': 0.8210665501105633, 'dropout_rate': 0.9557982366790785, 'n_estimators': 50, 'learning_rate': 0.03153701184110286}. Best is trial 56 with value: 0.22653496264114223.
Fold 1 IBS: 0.2450009034050451
Fold 2 IBS: 0.21574021216390366
Fold 3 IBS: 0.2316126494096215
Fold 4 IBS: 0.2313085066143514
Fold 5 IBS: 0.2135215585555951
[I 2024-04-19 20:13:15,895] Trial 59 finished with value: 0.22743676602970336 and parameters: {'subsample': 0.7

Fold 5 IBS: 0.21457551785343906
[I 2024-04-19 20:14:29,861] Trial 76 finished with value: 0.2278190214501346 and parameters: {'subsample': 0.5683268745859826, 'dropout_rate': 0.8653081987830238, 'n_estimators': 34, 'learning_rate': 0.020347121996093395}. Best is trial 56 with value: 0.22653496264114223.
Fold 1 IBS: 0.24975605957417094
Fold 2 IBS: 0.20544669926019324
Fold 3 IBS: 0.24292589881032364
Fold 4 IBS: 0.2283240329592914
Fold 5 IBS: 0.21054396005104126
[I 2024-04-19 20:14:34,002] Trial 77 finished with value: 0.22739933013100408 and parameters: {'subsample': 0.5096986308003991, 'dropout_rate': 0.49881327862855496, 'n_estimators': 145, 'learning_rate': 0.010267257888313383}. Best is trial 56 with value: 0.22653496264114223.
Fold 1 IBS: 0.24684927097070503
Fold 2 IBS: 0.2094952280230178
Fold 3 IBS: 0.23652719105524916
Fold 4 IBS: 0.22913951054643386
Fold 5 IBS: 0.2106573202990221
[I 2024-04-19 20:14:36,125] Trial 78 finished with value: 0.2265337041788856 and parameters: {'subsamp

Fold 5 IBS: 0.21018857248815617
[I 2024-04-19 20:15:26,341] Trial 95 finished with value: 0.22600775713838323 and parameters: {'subsample': 0.37798247189331685, 'dropout_rate': 0.8572208063796695, 'n_estimators': 112, 'learning_rate': 0.01063058308878333}. Best is trial 85 with value: 0.22596006011310088.
Fold 1 IBS: 0.2471425193657016
Fold 2 IBS: 0.20576111765918625
Fold 3 IBS: 0.2415748835282002
Fold 4 IBS: 0.22741639083301954
Fold 5 IBS: 0.209543331672624
[I 2024-04-19 20:15:29,850] Trial 96 finished with value: 0.22628764861174636 and parameters: {'subsample': 0.34523349906195283, 'dropout_rate': 0.7739280807288145, 'n_estimators': 132, 'learning_rate': 0.010648288131411126}. Best is trial 85 with value: 0.22596006011310088.
Fold 1 IBS: 0.24385115968166024
Fold 2 IBS: 0.21643646571599923
Fold 3 IBS: 0.23116557489710682
Fold 4 IBS: 0.23167317839356094
Fold 5 IBS: 0.21518657563145072
[I 2024-04-19 20:15:35,194] Trial 97 finished with value: 0.22766259086395557 and parameters: {'subsa

In [75]:
train_cindex['ComponentwiseGradientBoosting'] = np.round(study_cindex.best_value, 3)
train_ibs['ComponentwiseGradientBoosting'] = np.round(study_ibs.best_value, 3)

In [76]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.667
train_ibs:  0.224


#### Test

In [77]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [78]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(ComponentwiseGradientBoostingSurvivalAnalysis, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(ComponentwiseGradientBoostingSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

ComponentwiseGradientBoostingSurvivalAnalysis(dropout_rate=0.3060837787360696,
                                              learning_rate=0.04133902891970489,
                                              n_estimators=1, random_state=123,
                                              subsample=0.1001314126580015)

C-index score: 0.535


ComponentwiseGradientBoostingSurvivalAnalysis(dropout_rate=0.7782160113329367,
                                              learning_rate=0.005308151321564225,
                                              n_estimators=212,
                                              random_state=123,
                                              subsample=0.17778434918103095)

IBS: 0.233


In [79]:
# Saving the values to the dictionary 
test_cindex['ComponentwiseGradientBoosting'] = c_index
test_ibs['ComponentwiseGradientBoosting'] = ibs

## Results

In [80]:
df_train_cindex = pd.DataFrame(train_cindex, index=['C-index']).transpose().sort_values(by='C-index', ascending=False)
df_train_cindex['rank'] = df_train_cindex['C-index'].rank(ascending=False)
df_train_cindex 

,C-index,rank
Randomsurvivalforest,0.866,1.0
ExtraSurvivalTrees,0.826,2.0
GradientBoosting,0.759,3.0
CoxElastic,0.726,4.0
CoxLasso,0.725,5.0
CoxPH,0.724,6.0
ComponentwiseGradientBoosting,0.667,7.0
CoxRidge,0.651,8.0


In [81]:
df_train_ibs = pd.DataFrame(train_ibs, index=['IBS']).transpose().sort_values(by='IBS', ascending=True)
df_train_ibs['rank'] = df_train_ibs['IBS'].rank(ascending=True)
df_train_ibs

,IBS,rank
ExtraSurvivalTrees,0.188,1.0
Randomsurvivalforest,0.190,2.0
CoxLasso,0.199,3.5
CoxElastic,0.199,3.5
CoxPH,0.200,5.0
ComponentwiseGradientBoosting,0.224,6.0
GradientBoosting,0.233,7.0
CoxRidge,0.236,8.0


In [82]:
df_test_cindex = pd.DataFrame(test_cindex, index=['C-index']).transpose().sort_values(by='C-index', ascending=False)
df_test_cindex['rank'] = df_test_cindex['C-index'].rank(ascending=False)
df_test_cindex 

,C-index,rank
ExtraSurvivalTrees,0.607,1.0
GradientBoosting,0.586,2.0
CoxPH,0.562,3.0
CoxLasso,0.561,4.0
CoxElastic,0.560,5.0
Randomsurvivalforest,0.546,6.0
CoxRidge,0.535,7.5
ComponentwiseGradientBoosting,0.535,7.5


In [83]:
df_test_ibs = pd.DataFrame(test_ibs, index=['IBS']).transpose().sort_values(by='IBS', ascending=True)
df_test_ibs['rank'] = df_test_ibs['IBS'].rank(ascending=True)
df_test_ibs 

,IBS,rank
GradientBoosting,0.228,1.0
CoxRidge,0.229,2.0
ComponentwiseGradientBoosting,0.233,3.0
ExtraSurvivalTrees,0.252,4.0
Randomsurvivalforest,0.263,5.0
CoxLasso,0.288,6.0
CoxElastic,0.289,7.0
CoxPH,0.290,8.0


In [84]:
# Renaming the column "index" to "model" 
df_train_cindex = df_train_cindex.reset_index().rename(columns={"index": "model"})
df_train_ibs = df_train_ibs.reset_index().rename(columns={"index": "model"})
df_test_cindex = df_test_cindex.reset_index().rename(columns={"index": "model"})
df_test_ibs = df_test_ibs.reset_index().rename(columns={"index": "model"})

# Save the files 
dfs = [df_train_cindex, df_train_ibs, df_test_cindex, df_test_ibs]  
file_path = 'path_to_your_folder/'  

# List of corresponding file names
file_names = ['train_cindex.csv', 'train_ibs.csv', 'test_cindex.csv', 'test_ibs.csv']


dfs = [df_train_cindex, df_train_ibs, df_test_cindex, df_test_ibs]  
file_path = '/Users/minjeongcheon/Desktop/results_thesis/d3/dfs/robust/rent/'  

# List of corresponding file names
file_names = ['train_cindex.csv', 'train_ibs.csv', 'test_cindex.csv', 'test_ibs.csv']

# Modify the file names to match the desired format
modified_file_names = ['d3_dfs_robust_rent_' + file_name for file_name in file_names]

# Loop through each DataFrame and save them with corresponding modified file names
for df, modified_file_name in zip(dfs, modified_file_names):
    file_path_name = file_path + modified_file_name  # Construct the full file path
    df.to_csv(file_path_name, index=False)  # Save the DataFrame to CSV file


In [85]:
from datetime import date
today = date.today()
print("Date: ", today)

Date:  2024-04-19
